In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

In [2]:
from dotenv import load_dotenv
import os

load_dotenv() # to load my openai api key

True

In [3]:
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "streamable_http",
            "url": "http://localhost:3001/mcp" # FDE MCP server endpoint
        },
    }
)

In [4]:
tools = await client.get_tools()
tools

[StructuredTool(name='search-internal-watchlist', description="Search the application's internal watchlist for a given entity to detect fraud or risk", args_schema={'type': 'object', 'properties': {'entityType': {'type': 'string', 'enum': ['SSN', 'TIN', 'ACCOUNT_MOBILE', 'EIN', 'ACCOUNT_EMAIL', 'ACCOUNT_NAME', 'IP_ADDRESS'], 'description': 'The type of entity to search for in the internal watchlist which could be SSN, TIN, ACCOUNT_MOBILE, EIN, ACCOUNT_EMAIL, ACCOUNT_NAME, IP_ADDRESS'}, 'entityValue': {'type': 'string', 'description': 'The value of the entity to search for in the internal watchlist'}}, 'required': ['entityType', 'entityValue'], 'additionalProperties': False, '$schema': 'http://json-schema.org/draft-07/schema#'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11b5d9300>),
 StructuredTool(name='search-external-watchlist', description="Search the application's external watchlist for a given entity to d

In [6]:
agent = create_react_agent("openai:gpt-4.1", tools)

In [23]:
response = await agent.ainvoke({"messages": """
Does the user appear in the internal watchlist or external watchlist? 
The user's name is edward fitzgibbon, thier ssn is 216689350, phone number is +19823376358 and their email address is edward@outlook.com"""})

In [24]:
response

{'messages': [HumanMessage(content="\nDoes the user appear in the internal watchlist or external watchlist? \nThe user's name is edward fitzgibbon, thier ssn is 216689350, phone number is +19823376358 and their email address is edward@outlook.com", additional_kwargs={}, response_metadata={}, id='8e2a325f-1099-4228-91cd-03322c556aa9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_QBQ89MIRgSIbJUDqaekWYFsX', 'function': {'arguments': '{"entityType": "ACCOUNT_NAME", "entityValue": "edward fitzgibbon"}', 'name': 'search-internal-watchlist'}, 'type': 'function'}, {'id': 'call_z9DpMOlbJRJ2DMNdKi8atcxG', 'function': {'arguments': '{"entityType": "SSN", "entityValue": "216689350"}', 'name': 'search-internal-watchlist'}, 'type': 'function'}, {'id': 'call_ifzoFBnPaJ89VZckBEUpvtwA', 'function': {'arguments': '{"entityType": "ACCOUNT_MOBILE", "entityValue": "+19823376358"}', 'name': 'search-internal-watchlist'}, 'type': 'function'}, {'id': 'call_8rBGXokNy1GeuTNMQ4yb1Hbq', 

In [ ]:


from langchain_core.messages import AIMessage


ai_messages = [ai_message for ai_message in response.get("messages", []) if isinstance(ai_message, AIMessage)]
print(ai_messages[1].content)


The user "edward fitzgibbon" appears in both the internal and external watchlists:

Internal Watchlist:
- Name ("edward fitzgibbon"): Listed and blocked from application by DLAP risk review.
- SSN (216689350): Listed and blocked from application by DLAP risk review.
- Phone Number (+19823376358): Listed and blocked by DLAP/CLS risk review.
- Email (edward@outlook.com): Listed and blocked from application by DLAP risk review.

External Watchlist:
- SSN (216689350): Listed in the external watchlist (LIRA system) for a TIN/SSN mismatch involving this user.

Summary: All user-provided details are on the internal watchlist, and the SSN also appears on the external watchlist with a high-risk note. This indicates multiple risk flags across systems.
